# ⏳ AutoTimeTree: Interactive Jupyter Notebook Pipeline

Welcome to **AutoTimeTree**! This notebook allows you to run the complete molecular clock and time tree analysis pipeline interactively step-by-step.

### Pipeline Steps:
1. **Sequence Cleaning & MAFFT Alignment**
2. **Maximum Likelihood (FastTree) & Time Tree Chronogram (`ape::chronos`)**
3. **Pairwise p-distance Matrix Computation**
4. **Publication Multi-Panel Figure Rendering**
5. **Automated Word Document Report Generation**

## ⚙️ Step 0: Imports & Environment Setup

In [ ]:
import os
import sys
import subprocess
import numpy as np
import pandas as pd
from Bio import SeqIO, Phylo
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import seaborn as sns

# Ensure results directory exists
os.makedirs("results", exist_ok=True)

# Ensure binary tools (mafft, fasttree) are in PATH
os.environ["PATH"] = "/opt/miniconda3/bin:/usr/local/bin:/opt/homebrew/bin:" + os.environ.get("PATH", "")

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = 'Helvetica'
print("[+] AutoTimeTree environment initialized successfully.")

## ⚡ Step 1: Sequence Cleaning & MAFFT Alignment

In [ ]:
input_fasta = "data/example_input.fasta"
cleaned_fasta = "results/cleaned_sequences.fasta"
aligned_fasta = "results/aligned_sequences.fasta"

# Clean FASTA headers
records = []
with open(input_fasta, 'r') as f:
    header = None
    seq_lines = []
    for line in f:
        line = line.strip()
        if not line: continue
        if line.startswith('>'):
            if header: records.append((header, ''.join(seq_lines)))
            header = line[1:].strip().replace(' ', '_').replace('.', '_')
            seq_lines = []
        else:
            seq_lines.append(line)
    if header: records.append((header, ''.join(seq_lines)))

with open(cleaned_fasta, 'w') as out_f:
    for h, s in records:
        out_f.write(f">{h}\n{s}\n")

print(f"[+] Saved {len(records)} cleaned sequences to '{cleaned_fasta}'.")

# Run MAFFT alignment
print("[*] Running MAFFT alignment...")
subprocess.run(f"mafft --auto '{cleaned_fasta}' > '{aligned_fasta}'", shell=True, check=True)
print(f"[+] MAFFT alignment complete: '{aligned_fasta}'.")

## 🌲 Step 2: Maximum Likelihood & Time Tree Chronogram Construction

In [ ]:
outgroup_keyword = "OUTGROUP"
ml_tree_file = "results/ml_tree.nwk"
rooted_tree_file = "results/rooted_ml_tree.nwk"
timetree_file = "results/timetree.nwk"

r_script = f"""
library(ape)
system("fasttree -gtr -nt '{aligned_fasta}' > '{ml_tree_file}'")
tree <- read.tree('{ml_tree_file}')
outgroup_tips <- grep('{outgroup_keyword}', tree[['tip.label']], value = TRUE, ignore.case = TRUE)
if (length(outgroup_tips) > 0) {{
  tree <- root(tree, outgroup = outgroup_tips[1], resolve.root = TRUE)
}}
write.tree(tree, '{rooted_tree_file}')
tree[['node.label']] <- NULL
tree[['edge.length']][tree[['edge.length']] < 1e-6] <- 1e-6
time_tree <- chronos(tree, lambda = 1, model = 'relaxed')
write.tree(time_tree, '{timetree_file}')
cat('[+] R execution complete.\n')
"""

with open("results/temp_run.R", "w") as f:
    f.write(r_script)

subprocess.run("Rscript results/temp_run.R", shell=True, check=True)
print("[+] ML tree and Time Tree generated in results/")

## 📊 Step 3: Compute Pairwise Distance Matrix & Generate Publication Plot

In [ ]:
alignment = list(SeqIO.parse(aligned_fasta, 'fasta'))
seq_dict = {rec.id: str(rec.seq).upper() for rec in alignment}
taxa = list(seq_dict.keys())
n_taxa = len(taxa)

def get_group(t):
    if outgroup_keyword.lower() in t.lower() or 'xylella' in t.lower(): return 'Outgroup'
    elif t.endswith('_P'): return 'Pathogenic'
    elif t.endswith('_NP'): return 'Non-Pathogenic'
    else: return 'Intermediate'

group_map = {t: get_group(t) for t in taxa}
color_map = {'Pathogenic': '#D9381E', 'Non-Pathogenic': '#0072B2', 'Outgroup': '#555555', 'Intermediate': '#E69F00'}

dist_matrix = np.zeros((n_taxa, n_taxa))
for i in range(n_taxa):
    for j in range(i+1, n_taxa):
        s1, s2 = seq_dict[taxa[i]], seq_dict[taxa[j]]
        valid = sum(1 for a, b in zip(s1, s2) if a in 'ATCG' and b in 'ATCG')
        diffs = sum(1 for a, b in zip(s1, s2) if a in 'ATCG' and b in 'ATCG' and a != b)
        p_dist = diffs / valid if valid > 0 else 0.0
        dist_matrix[i, j] = dist_matrix[j, i] = p_dist

df_dist = pd.DataFrame(dist_matrix, index=taxa, columns=taxa)
df_dist.to_csv("results/pairwise_pdistances.csv")
print("[+] Distance matrix saved to 'results/pairwise_pdistances.csv'.")

# Plot Multi-Panel Figure
ml_tree = Phylo.read(rooted_tree_file, 'newick')
time_tree = Phylo.read(timetree_file, 'newick')

fig = plt.figure(figsize=(16, 12))
gs = GridSpec(2, 2, figure=fig, width_ratios=[1.2, 1], height_ratios=[1, 0.8], hspace=0.3, wspace=0.25)

def draw_tree(tree, ax, title, is_timetree=False):
    terminals = tree.get_terminals()
    ml_order = [c.name for c in ml_tree.get_terminals()]
    terminals = sorted(terminals, key=lambda c: ml_order.index(c.name) if c.name in ml_order else 0)
    y_coords = {c: i for i, c in enumerate(terminals)}
    def get_x(c): return tree.distance(tree.root, c)
    def draw_clade(clade):
        x_parent = get_x(clade)
        if clade.is_terminal():
            y_parent = y_coords[clade]
            c_color = color_map.get(group_map.get(clade.name, 'Intermediate'), '#555555')
            ax.scatter(x_parent, y_parent, color=c_color, s=35, zorder=5)
            ax.text(x_parent + 0.01, y_parent, clade.name, va='center', fontsize=7, fontweight='bold', color=c_color)
            return y_parent
        else:
            child_ys = [draw_clade(child) for child in clade]
            y_min, y_max = min(child_ys), max(child_ys)
            ax.plot([x_parent, x_parent], [y_min, y_max], color='#444444', lw=1.2)
            for child in clade:
                x_child = get_x(child)
                y_child = child_ys[clade.clades.index(child)]
                ax.plot([x_parent, x_child], [y_child, y_child], color='#444444', lw=1.2)
            return sum(child_ys) / len(child_ys)
    draw_clade(tree.root)
    ax.set_ylim(-1, len(terminals))
    ax.set_yticks([])
    ax.set_title(title, fontsize=11, fontweight='bold')

ax_tt = fig.add_subplot(gs[0, 0])
draw_tree(time_tree, ax_tt, 'A. Time Tree Chronogram (Relaxed Molecular Clock)', is_timetree=True)

ax_ml = fig.add_subplot(gs[0, 1])
draw_tree(ml_tree, ax_ml, 'B. Maximum Likelihood Phylogram', is_timetree=False)

ax_hm = fig.add_subplot(gs[1, 0])
sns.heatmap(df_dist, ax=ax_hm, cmap='YlOrRd', cbar_kws={'label': 'p-distance'})
ax_hm.set_title('C. Pairwise Distance Heatmap', fontsize=11, fontweight='bold')
ax_hm.tick_params(axis='both', labelsize=6)

ax_box = fig.add_subplot(gs[1, 1])
p_taxa = [t for t in taxa if group_map[t] == 'Pathogenic']
np_taxa = [t for t in taxa if group_map[t] == 'Non-Pathogenic']
p_p_dists = [df_dist.loc[t1, t2] for i, t1 in enumerate(p_taxa) for t2 in p_taxa[i+1:]]
np_np_dists = [df_dist.loc[t1, t2] for i, t1 in enumerate(np_taxa) for t2 in np_taxa[i+1:]]
p_np_dists = [df_dist.loc[t1, t2] for t1 in p_taxa for t2 in np_taxa]

data_box = [{'Group': 'Path (Intra)', 'p-distance': d} for d in p_p_dists] + \
           [{'Group': 'Non-Path (Intra)', 'p-distance': d} for d in np_np_dists] + \
           [{'Group': 'Inter-group', 'p-distance': d} for d in p_np_dists]
df_box = pd.DataFrame(data_box)
sns.boxplot(data=df_box, x='Group', y='p-distance', ax=ax_box, palette=['#D9381E', '#0072B2', '#8E44AD'])
ax_box.set_title('D. Group Divergence Distribution', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig("results/timetree_analysis_multipanel.png", dpi=300)
plt.show()

## 📄 Step 4: Generate Word (.docx) Report

In [ ]:
try:
    from docx import Document
    from docx.shared import Inches, Pt, RGBColor
    doc = Document()
    doc.add_heading('AutoTimeTree: Molecular Clock Report', level=0)
    doc.add_paragraph('Automated report generated from AutoTimeTree Jupyter Notebook.')
    if os.path.exists('results/timetree_analysis_multipanel.png'):
        doc.add_picture('results/timetree_analysis_multipanel.png', width=Inches(6.0))
    doc.save('results/Evolutionary_TimeTree_Report.docx')
    print("[+] Word document report saved successfully to 'results/Evolutionary_TimeTree_Report.docx'.")
except ImportError:
    print("[!] python-docx is not installed. Install with 'pip install python-docx'.")